# Clase 197 — CI/CD para ML con GitHub Actions

Notebook **declarativo**: muestra los YAML de workflow y scripts del repo objetivo. Para verlos en vivo, copialos a un repo GitHub y abrí un PR.

Estructura del repo:
```
.github/workflows/ml.yml
src/train.py · src/evaluate.py · src/diff_metrics.py · src/check_threshold.py
tests/test_train.py
params.yaml · requirements.txt
```

## 1. Workflow `ml.yml`

In [ ]:
workflow = '''\
name: ml
on:
  pull_request: { branches: [main] }
  push: { branches: [main] }
permissions:
  contents: read
  pull-requests: write
  id-token: write   # OIDC para AWS
jobs:
  lint:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.12", cache: pip }
      - run: pip install ruff && ruff check src tests
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.12", cache: pip }
      - run: pip install -r requirements.txt && pytest -q
  train-and-report:
    if: github.event_name == 'pull_request'
    runs-on: ubuntu-latest
    needs: [lint, test]
    steps:
      - uses: actions/checkout@v4
        with: { fetch-depth: 0 }
      - uses: actions/setup-python@v5
        with: { python-version: "3.12", cache: pip }
      - uses: iterative/setup-cml@v2
      - run: pip install -r requirements.txt
      - run: python src/train.py && python src/evaluate.py > metrics_pr.json
      - name: Train on main for comparison
        run: |
          git checkout origin/main -- src/
          python src/train.py && python src/evaluate.py > metrics_main.json
          git checkout HEAD -- src/
      - name: Comment report
        env: { REPO_TOKEN: "${{ secrets.GITHUB_TOKEN }}" }
        run: |
          python src/diff_metrics.py metrics_main.json metrics_pr.json > report.md
          cml comment create report.md
      - run: python src/check_threshold.py metrics_main.json metrics_pr.json 0.03
'''
print(workflow)

## 2. `diff_metrics.py` y `check_threshold.py`

In [ ]:
diff_metrics = '''\
import json, sys
main, pr = json.load(open(sys.argv[1])), json.load(open(sys.argv[2]))
print("## 📊 Modelo PR vs main\\n")
print("| métrica | main | PR | Δ |\\n|---|---|---|---|")
for k in sorted(set(main) | set(pr)):
    a, b = main.get(k, 0), pr.get(k, 0)
    print(f"| {k} | {a:.4f} | {b:.4f} | {'🟢' if b >= a else '🔴'} {b - a:+.4f} |")
'''
check = '''\
import json, sys
main, pr, tol = json.load(open(sys.argv[1])), json.load(open(sys.argv[2])), float(sys.argv[3])
regr = [f"{k}: {main[k]:.4f} -> {pr[k]:.4f}" for k in main if k in pr and pr[k] < main[k] - tol]
if regr: print("REGRESSION:", *regr, sep="\\n"); sys.exit(1)
print("OK")
'''
print(diff_metrics)
print('---')
print(check)

## 3. Simulación local: 2 modelos comparados

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

X, y = load_iris(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)

def eval_model(m, name):
    p = m.fit(Xtr, ytr).predict(Xte)
    return {'model': name, 'accuracy': accuracy_score(yte, p), 'f1_macro': f1_score(yte, p, average='macro')}

main_m = eval_model(RandomForestClassifier(n_estimators=100, random_state=42), 'rf-100')
pr_m = eval_model(LogisticRegression(max_iter=500), 'lr')
print('main:', main_m); print('pr:  ', pr_m)

print('\n## Δ vs main')
for k in ['accuracy', 'f1_macro']:
    d = pr_m[k] - main_m[k]
    print(f'  {k}: {d:+.4f}  {"🟢" if d >= 0 else "🔴"}')

## Ejercicio guiado

1. Copiá el workflow a un repo real. Hacé un PR con `RandomForest(n_estimators=5)` (peor). Confirmá comment CML + check rojo.
2. Activá branch protection en `main` con `lint`, `test`, `train-and-report` como required.
3. Bonus: job `deploy` solo en `push: main`, con OIDC a AWS y `aws s3 cp model.pkl s3://bucket/`.

## Conclusiones

- CI/CD convierte "el modelo nuevo es mejor" en un check ejecutable.
- Cache de pip + datasets corta CI de 25 min a 3 min.
- OIDC reemplaza secrets long-lived.
- Sin branch protection + required checks, el CI es sugerencia, no gate.